# Trying collaborator suggestions for premise selection

In [ ]:
from pathlib import Path
import sys
import os

current_directory = Path(os.getcwd())
print(current_directory)
root_directory = current_directory.parent.parent

sys.path.append(str(root_directory))

In [ ]:
from pprint import pprint
import pickle
from src.llm import (
    ChatMessage,
    AssistantMessage,
    UserMessage,
    SystemMessage,
    ChatTrace,
)
from src.llm.gpt import (
    OpenaiToolCall,
    OpenaiChatPromptConfig,
    OpenaiToolConfig,
    OpenaiTool,
    chat, chat_with_tools, rate_limited_chat
)
import typing as t
import random

In [ ]:
# dataset of (proposition, proof state, lemmas)
DATASET = [
    (
        "wmonotonic TX TX G",
        """G : forall _ : relation2 X X, relation2 X X
F : forall _ : relation2 X X, relation2 X X
TX : reduction_t A X
A,X : Type

---

wmonotonic TX TX G""",
        """BinNat.N.Even_Odd_False: forall (x : BinNums.N) (_ : BinNat.N.Even x) (_ : BinNat.N.Odd x), False

BoolSpecF: forall (P Q : Prop) (_ : Q), BoolSpec P Q false

Chaining_wmon: forall (A X : Type) (TX : reduction_t A X) (F G : function X X) (_ : wmonotonic TX TX F) (_ : wmonotonic TX TX G), wmonotonic TX TX (Chain F G)

Comp_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F G : function X Y) (_ : wmonotonic TX TY F) (_ : wmonotonic TX TY G), wmonotonic TX TY (Comp G F)

EnvRing.MFactor: forall (C : Type) (_ : C) (_ : forall (_ : C) (_ : C), bool) (_ : EnvRing.Pol C) (_ : EnvRing.Mon), prod (EnvRing.Pol C) (EnvRing.Pol C)

False_ind: forall (P : Prop) (_ : False), P

Field_theory.AF_AR: forall (R : Type) (rO rI : R) (radd rmul rsub : forall (_ : R) (_ : R), R) (ropp : forall _ : R, R) (rdiv : forall (_ : R) (_ : R), R) (rinv : forall _ : R, R) (req : forall (_ : R) (_ : R), Prop) (_ : Field_theory.almost_field_theory rO rI radd rmul rsub ropp rdiv rinv req), almost_ring_theory rO rI radd rmul rsub ropp req

Field_theory.FEadd: forall (C : Type) (_ : Field_theory.FExpr C) (_ : Field_theory.FExpr C), Field_theory.FExpr C

Field_theory.FEc: forall (C : Type) (_ : C), Field_theory.FExpr C

Field_theory.FEpow: forall (C : Type) (_ : Field_theory.FExpr C) (_ : BinNums.N), Field_theory.FExpr C

Field_theory.FExpr_rect: forall (C : Type) (P : forall _ : Field_theory.FExpr C, Type) (_ : P Field_theory.FEO) (_ : P Field_theory.FEI) (_ : forall c : C, P (Field_theory.FEc c)) (_ : forall p : BinNums.positive, P (Field_theory.FEX C p)) (_ : forall (f3 : Field_theory.FExpr C) (_ : P f3) (f4 : Field_theory.FExpr C) (_ : P f4), P (Field_theory.FEadd f3 f4)) (_ : forall (f4 : Field_theory.FExpr C) (_ : P f4) (f5 : Field_theory.FExpr C) (_ : P f5), P (Field_theory.FEsub f4 f5)) (_ : forall (f5 : Field_theory.FExpr C) (_ : P f5) (f6 : Field_theory.FExpr C) (_ : P f6), P (Field_theory.FEmul f5 f6)) (_ : forall (f6 : Field_theory.FExpr C) (_ : P f6), P (Field_theory.FEopp f6)) (_ : forall (f7 : Field_theory.FExpr C) (_ : P f7), P (Field_theory.FEinv f7)) (_ : forall (f8 : Field_theory.FExpr C) (_ : P f8) (f9 : Field_theory.FExpr C) (_ : P f9), P (Field_theory.FEdiv f8 f9)) (_ : forall (f9 : Field_theory.FExpr C) (_ : P f9) (n : BinNums.N), P (Field_theory.FEpow f9 n)) (f10 : Field_theory.FExpr C), P f10

Field_theory.Pcond_Fnorm: forall (R : Type) (rO rI : R) (radd rmul rsub : forall (_ : R) (_ : R), R) (ropp : forall _ : R, R) (rdiv : forall (_ : R) (_ : R), R) (rinv : forall _ : R, R) (req : forall (_ : R) (_ : R), Prop) (_ : RelationClasses.Equivalence req) (_ : ring_eq_ext radd rmul ropp req) (_ : Field_theory.almost_field_theory rO rI radd rmul rsub ropp rdiv rinv req) (C : Type) (cO cI : C) (cadd cmul csub : forall (_ : C) (_ : C), C) (copp : forall _ : C, C) (ceqb : forall (_ : C) (_ : C), bool) (phi : forall _ : C, R) (_ : ring_morph rO rI radd rmul rsub ropp req cO cI cadd cmul csub copp ceqb phi) (Cpow : Type) (Cp_phi : forall _ : BinNums.N, Cpow) (rpow : forall (_ : R) (_ : Cpow), R) (_ : power_theory rI rmul req Cp_phi rpow) (l : list R) (e : Field_theory.FExpr C) (_ : Field_theory.PCond rO rI radd rmul rsub ropp req phi Cp_phi rpow l (Field_theory.condition (Field_theory.Fnorm cO cI cadd cmul csub copp ceqb e))), not (req (Ring_polynom.PEeval rO rI radd rmul rsub ropp phi Cp_phi rpow l (Field_theory.denum (Field_theory.Fnorm cO cI cadd cmul csub copp ceqb e))) rO)

Field_theory.SFdiv_def: forall (R : Type) (rO rI : R) (radd rmul rdiv : forall (_ : R) (_ : R), R) (rinv : forall _ : R, R) (req : forall (_ : R) (_ : R), Prop) (_ : Field_theory.semi_field_theory rO rI radd rmul rdiv rinv req) (p q : R), req (rdiv p q) (rmul p (rinv q))

Fin.F1: forall n : nat, Fin.t (S n)

Fix_F_2: forall (A B : Type) (R : forall (_ : prod A B) (_ : prod A B), Prop) (P : forall (_ : A) (_ : B), Type) (_ : forall (x : A) (x' : B) (_ : forall (y : A) (y' : B) (_ : R (pair y y') (pair x x')), P y y'), P x x') (x : A) (x' : B) (_ : Acc R (pair x x')), P x x'

Fix_F_inv: forall (A : Type) (R : forall (_ : A) (_ : A), Prop) (_ : well_founded R) (P : forall _ : A, Type) (F : forall (x : A) (_ : forall (y : A) (_ : R y x), P y), P x) (_ : forall (x : A) (f g : forall (y : A) (_ : R y x), P y) (_ : forall (y : A) (p : R y x), eq (f y p) (g y p)), eq (F x f) (F x g)) (x : A) (r s : Acc R x), eq (Fix_F P F r) (Fix_F P F s)

Lemma F_mon: monotonic TX TX F.

List.Forall2: forall (A B : Type) (_ : forall (_ : A) (_ : B), Prop) (_ : list A) (_ : list B), Prop

List.Forall2_app: forall (A B : Type) (R : forall (_ : A) (_ : B), Prop) (l1 l2 : list A) (l1' l2' : list B) (_ : List.Forall2 R l1 l1') (_ : List.Forall2 R l2 l2'), List.Forall2 R (app l1 l2) (app l1' l2')

List.Forall2_app_inv_r: forall (A B : Type) (R : forall (_ : A) (_ : B), Prop) (l1' l2' : list B) (l : list A) (_ : List.Forall2 R l (app l1' l2')), ex (fun l1 : list A => ex (fun l2 : list A => and (List.Forall2 R l1 l1') (and (List.Forall2 R l2 l2') (eq l (app l1 l2)))))

List.Forall2_cons: forall (A B : Type) (R : forall (_ : A) (_ : B), Prop) (x : A) (y : B) (l : list A) (l' : list B) (_ : R x y) (_ : List.Forall2 R l l'), List.Forall2 R (cons x l) (cons y l')

List.Forall2_refl: forall (A B : Type) (R : forall (_ : A) (_ : B), Prop), List.Forall2 R nil nil

List.Forall: forall (A : Type) (_ : forall _ : A, Prop) (_ : list A), Prop

RingMicromega.map_Formula: forall (C S : Type) (_ : forall _ : S, C) (_ : RingMicromega.Formula S), RingMicromega.Formula C

Tauto.GFormula_rect: forall (TA TX AA AF : Type) (P : forall _ : Tauto.GFormula, Type) (_ : P Tauto.TT) (_ : P Tauto.FF) (_ : forall t : TX, P (Tauto.X t)) (_ : forall (t : TA) (a : AA), P (Tauto.A t a)) (_ : forall (g : Tauto.GFormula) (_ : P g) (g0 : Tauto.GFormula) (_ : P g0), P (Tauto.Cj g g0)) (_ : forall (g : Tauto.GFormula) (_ : P g) (g0 : Tauto.GFormula) (_ : P g0), P (Tauto.D g g0)) (_ : forall (g : Tauto.GFormula) (_ : P g), P (Tauto.N g)) (_ : forall (g : Tauto.GFormula) (_ : P g) (o : option AF) (g0 : Tauto.GFormula) (_ : P g0), P (Tauto.I g o g0)) (g : Tauto.GFormula), P g

UExp_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (n : nat), wmonotonic TX TY (fun R : relation2 X Y => UExp F R n)

UIter_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F), wmonotonic TX TY (UIter F)

Union2_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F G : function X Y) (_ : wmonotonic TX TY F) (_ : wmonotonic TX TY G), wmonotonic TX TY (Union2 F G)

Union_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (I : Type) (H : forall _ : I, function X Y) (_ : forall i : I, wmonotonic TX TY (H i)), wmonotonic TX TY (Union H)

Vector.Forall2: forall (A B : Type) (_ : forall (_ : A) (_ : B), Prop) (n : nat) (_ : Vector.t A n) (_ : Vector.t B n), Prop

Vector.Forall_nil: forall (A : Type) (P : forall _ : A, Prop), Vector.Forall P (Vector.nil A)

VectorDef.Forall2: forall (A B : Type) (_ : forall (_ : A) (_ : B), Prop) (n : nat) (_ : VectorDef.t A n) (_ : VectorDef.t B n), Prop

ZMicromega.Vars.Facts.MP.Dec.FSetDecideAuxiliary.FSet_Prop_ind: forall (P : forall _ : Prop, Prop) (_ : forall (P0 : Prop) (_ : ZMicromega.Vars.Facts.MP.Dec.FSetDecideAuxiliary.FSet_elt_Prop P0), P P0) (_ : forall s : FSetPositive.PositiveSet.t, P (FSetPositive.PositiveSet.Empty s)) (_ : forall s1 s2 : FSetPositive.PositiveSet.t, P (FSetPositive.PositiveSet.Subset s1 s2)) (_ : forall s1 s2 : FSetPositive.PositiveSet.t, P (FSetPositive.PositiveSet.Equal s1 s2)) (P0 : Prop) (_ : ZMicromega.Vars.Facts.MP.Dec.FSetDecideAuxiliary.FSet_Prop P0), P P0

chaing_l_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (L : relation Y) (_ : simulation TY TY L), wmonotonic TY TX (chaining_l L)

ctrl_a: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (B : relation X) (_ : controlled TX TY B) (R S : relation2 X Y) (_ : evolve_t TX TY R (comp (star B) R)) (_ : simulation_t TX TY S) (_ : evolve_a TX TY R S) (_ : incl R S), evolve_a TX TY (comp (star B) R) (comp (star B) S)

ctrl_t: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (B : relation X) (_ : controlled TX TY B) (R : relation2 X Y) (_ : evolve_t TX TY R (comp (star B) R)), simulation_t TX TY (comp (star B) R)

evolve_a: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : relation2 X Y) (_ : relation2 X Y), Prop

mkwmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : increasing F) (_ : forall (R : relation2 X Y) (_ : simulation_t TX TY R), simulation_t TX TY (F R)) (_ : forall (R S : relation2 X Y) (_ : simulation_t TX TY R) (_ : simulation_t TX TY S) (_ : evolve_a TX TY R S) (_ : incl R S), evolve_a TX TY (F R) (F S)), wmonotonic TX TY F

mon_a: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : monotonic TX TY F) (R S : relation2 X Y) (_ : evolve TX TY R S) (_ : incl R S), evolve_a TX TY (F R) (F S)

monotonic_wmonotonic: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : monotonic TX TY F), wmonotonic TX TY F

simulation_t: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : relation2 X Y), Prop

simulation_t_eeq: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (R S : relation2 X Y) (_ : eeq R S) (_ : simulation_t TX TY R), simulation_t TX TY S

ssrbool.addbF: ssrfun.right_id false ssrbool.addb

ssrbool.andbF: ssrfun.right_zero false andb

ssrbool.contraNF: forall (c b : bool) (_ : forall _ : is_true c, is_true b) (_ : is_true (negb b)), eq c false

ssrbool.introF: forall (P : Prop) (b : bool) (_ : Bool.reflect P b) (_ : not P), eq b false

star_wmon: forall (A X : Type) (TX : reduction_t A X), wmonotonic TX TX (star (X:=X))

wmon_a: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (R S : relation2 X Y) (_ : simulation_t TX TY R) (_ : simulation_t TX TY S) (_ : evolve_a TX TY R S) (_ : incl R S), evolve_a TX TY (F R) (F S)

wmon_t: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (R : relation2 X Y) (_ : simulation_t TX TY R), simulation_t TX TY (F R)

wmonotonic: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : function X Y), Prop

wmonotonic_correct: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (R : relation2 X Y) (_ : simulation_t TX TY R) (_ : evolve_a TX TY R (F R)), simulation TX TY (UIter F R)

wmonotonic_correct_t: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (R : relation2 X Y) (_ : simulation_t TX TY R), simulation_t TX TY (UIter F R)"""
    ),
    (
        "wmonotonic TX TX (Union2 (identity (Y:=X)) (constant (bisim TX TX)))",
        """G : forall _ : relation2 X X, relation2 X X
F : forall _ : relation2 X X, relation2 X X
TX : reduction_t A X
A,X : Type

---

wmonotonic TX TX (Union2 (identity (Y:=X)) (constant (bisim TX TX)))""",
        """Chaining_wmon: forall (A X : Type) (TX : reduction_t A X) (F G : function X X) (_ : wmonotonic TX TX F) (_ : wmonotonic TX TX G), wmonotonic TX TX (Chain F G)

Comp_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F G : function X Y) (_ : wmonotonic TX TY F) (_ : wmonotonic TX TY G), wmonotonic TX TY (Comp G F)

Datatypes.identity: forall (A : Type) (_ : A) (_ : A), Prop

Lemma F_mon: monotonic TX TX F.

UExp_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (n : nat), wmonotonic TX TY (fun R : relation2 X Y => UExp F R n)

UIter_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F), wmonotonic TX TY (UIter F)

Union2: forall (X Y X' Y' : Type) (_ : forall _ : relation2 X Y, relation2 X' Y') (_ : forall _ : relation2 X Y, relation2 X' Y'), function2 X Y X' Y'

Union2_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F G : function X Y) (_ : monotonic TX TY F) (_ : monotonic TX TY G), monotonic TX TY (Union2 F G)

Union2_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F G : function X Y) (_ : wmonotonic TX TY F) (_ : wmonotonic TX TY G), wmonotonic TX TY (Union2 F G)

Union_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (I : Type) (H : forall _ : I, function X Y) (_ : forall i : I, wmonotonic TX TY (H i)), wmonotonic TX TY (Union H)

ZMicromega.Vars.Facts.MP.fold_identity: forall s : FSetPositive.PositiveSet.t, FSetPositive.PositiveSet.Equal (FSetPositive.PositiveSet.fold FSetPositive.PositiveSet.add s FSetPositive.PositiveSet.empty) s

bisim: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : X) (_ : Y), Prop

bisim_refl: forall (A X : Type) (TX : reduction_t A X), reflexive (bisim TX TX)

bisim_sym: forall (A X : Type) (TX : reduction_t A X), symmetric (bisim TX TX)

bisim_trans: forall (A X : Type) (TX : reduction_t A X), transitive (bisim TX TX)

bisimulation: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : relation2 X Y), Prop

bisimulation_bisim: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y), bisimulation TX TY (bisim TX TY)

bisimulation_comp: forall (A X Y Z : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (TZ : reduction_t A Z) (R : relation2 X Y) (S : relation2 Y Z) (_ : bisimulation TX TY R) (_ : bisimulation TY TZ S), bisimulation TX TZ (comp R S)

chaing_l_wmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (L : relation Y) (_ : simulation TY TY L), wmonotonic TY TX (chaining_l L)

constant: forall (X Y X' Y' : Type) (_ : relation2 X' Y'), function2 X Y X' Y'

constant_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (R : relation2 X Y) (_ : simulation TX TY R), monotonic TX TY (constant R)

identity: forall X Y : Type, function X Y

identity_congr: forall (A B : Type) (f : forall _ : A, B) (x y : A) (_ : Datatypes.identity x y), Datatypes.identity (f x) (f y)

identity_ind: forall (A : Type) (a : A) (P : forall (a0 : A) (_ : Datatypes.identity a a0), Prop) (_ : P a (identity_refl a)) (y : A) (i : Datatypes.identity a y), P y i

identity_ind_r: forall (A : Type) (a : A) (P : forall _ : A, Prop) (_ : P a) (y : A) (_ : Datatypes.identity y a), P y

identity_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y), monotonic TX TY (identity (Y:=Y))

identity_rec: forall (A : Type) (a : A) (P : forall (a0 : A) (_ : Datatypes.identity a a0), Set) (_ : P a (identity_refl a)) (y : A) (i : Datatypes.identity a y), P y i

identity_rec_r: forall (A : Type) (a : A) (P : forall _ : A, Set) (_ : P a) (y : A) (_ : Datatypes.identity y a), P y

identity_rect: forall (A : Type) (a : A) (P : forall (a0 : A) (_ : Datatypes.identity a a0), Type) (_ : P a (identity_refl a)) (y : A) (i : Datatypes.identity a y), P y i

identity_rect_r: forall (A : Type) (a : A) (P : forall _ : A, Type) (_ : P a) (y : A) (_ : Datatypes.identity y a), P y

identity_refl: forall (A : Type) (a : A), Datatypes.identity a a

identity_sind: forall (A : Type) (a : A) (P : forall (a0 : A) (_ : Datatypes.identity a a0), SProp) (_ : P a (identity_refl a)) (y : A) (i : Datatypes.identity a y), P y i

identity_sym: forall (A : Type) (x y : A) (_ : Datatypes.identity x y), Datatypes.identity y x

identity_trans: forall (A : Type) (x y z : A) (_ : Datatypes.identity x y) (_ : Datatypes.identity y z), Datatypes.identity x z

mkwmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : increasing F) (_ : forall (R : relation2 X Y) (_ : simulation_t TX TY R), simulation_t TX TY (F R)) (_ : forall (R S : relation2 X Y) (_ : simulation_t TX TY R) (_ : simulation_t TX TY S) (_ : evolve_a TX TY R S) (_ : incl R S), evolve_a TX TY (F R) (F S)), wmonotonic TX TY F

monotonic_wmonotonic: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : monotonic TX TY F), wmonotonic TX TY F

not_identity_sym: forall (A : Type) (x y : A) (_ : notT (Datatypes.identity x y)), notT (Datatypes.identity y x)

star_wmon: forall (A X : Type) (TX : reduction_t A X), wmonotonic TX TX (star (X:=X))

wexpand_bisim: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y), incl (wexpand TX TY) (bisim TX TY)

wmonotonic: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : function X Y), Prop

wmonotonic_correct: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (R : relation2 X Y) (_ : simulation_t TX TY R) (_ : evolve_a TX TY R (F R)), simulation TX TY (UIter F R)

wmonotonic_correct_t: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (R : relation2 X Y) (_ : simulation_t TX TY R), simulation_t TX TY (UIter F R)"""
    ),
    (
        "monotonic TX TX F",
        """G : forall _ : relation2 X X, relation2 X X
F : forall _ : relation2 X X, relation2 X X
TX : reduction_t A X
A,X : Type

---

monotonic TX TX F""",
"""BinInt.Z.lt_le_incl: forall (n m : BinNums.Z) (_ : BinInt.Z.lt n m), BinInt.Z.le n m

BinNat.N.eq_le_incl: forall (n m : BinNums.N) (_ : eq n m), BinNat.N.le n m

Chain: forall (X Y X' Y' Z' : Type) (_ : function2 X Y X' Y') (_ : function2 X Y Y' Z') (_ : relation2 X Y), relation2 X' Z'

Comp: forall (X Y X' Y' X'' Y'' : Type) (_ : function2 X' Y' X'' Y'') (_ : function2 X Y X' Y') (_ : relation2 X Y), relation2 X'' Y''

Comp_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F G : function X Y) (_ : monotonic TX TY F) (_ : monotonic TX TY G), monotonic TX TY (Comp G F)

EWeak: forall (A X : Type) (_ : reduction_t A X), reduction_t A X

Exp: forall (X Y : Type) (_ : function X Y) (_ : relation2 X Y) (_ : nat), relation2 X Y

F: forall _ : relation2 X X, relation2 X X

G: forall _ : relation2 X X, relation2 X X

Iter: forall (X Y : Type) (_ : function X Y) (_ : relation2 X Y), relation2 X Y

List.NoDup_incl_length: forall (A : Type) (l l' : list A) (_ : List.NoDup l) (_ : List.incl l l'), le (length l) (length l')

List.incl: forall (A : Type) (_ : list A) (_ : list A), Prop

List.incl_Add_inv: forall (A : Type) (a : A) (l u v : list A) (_ : not (List.In a l)) (_ : List.incl (cons a l) v) (_ : List.Add a u v), List.incl l u

List.incl_app: forall (A : Type) (l m n : list A) (_ : List.incl l n) (_ : List.incl m n), List.incl (app l m) n

List.incl_appl: forall (A : Type) (l m n : list A) (_ : List.incl l n), List.incl l (app n m)

List.incl_appr: forall (A : Type) (l m n : list A) (_ : List.incl l n), List.incl l (app m n)

List.incl_cons: forall (A : Type) (a : A) (l m : list A) (_ : List.In a m) (_ : List.incl l m), List.incl (cons a l) m

List.incl_refl: forall (A : Type) (l : list A), List.incl l l

List.incl_tl: forall (A : Type) (a : A) (l m : list A) (_ : List.incl l m), List.incl l (cons a m)

List.incl_tran: forall (A : Type) (l m n : list A) (_ : List.incl l m) (_ : List.incl m n), List.incl l n

Listing only lemmas with conclusion matching (evolve_t _ _ _ _)

Listing only lemmas with conclusion matching (monotonic _ _ _)

Listing only lemmas with conclusion matching (reduction_t _ _)

Listing only lemmas with conclusion matching (relation2 _ _)

PeanoNat.Nat.eq_le_incl: forall (n m : nat) (_ : eq n m), le n m

REWeak: forall (A X : Type) (_ : reduction_t A X), reduction_t A X

Relation_Definitions.inclusion: forall (A : Type) (_ : Relation_Definitions.relation A) (_ : Relation_Definitions.relation A), Prop

SetoidList.ForallOrdPairs_inclA: forall (A : Type) (eqA : forall (_ : A) (_ : A), Prop) (_ : RelationClasses.Equivalence eqA) (R : forall (_ : A) (_ : A), Prop) (_ : RelationClasses.Symmetric R) (_ : Morphisms.Proper (Morphisms.respectful eqA (Morphisms.respectful eqA iff)) R) (l l' : list A) (_ : SetoidList.NoDupA eqA l') (_ : SetoidList.inclA eqA l' l) (_ : List.ForallOrdPairs R l), List.ForallOrdPairs R l'

SetoidList.inclA: forall (A : Type) (_ : forall (_ : A) (_ : A), Prop) (_ : list A) (_ : list A), Prop

SetoidList.incl_nil: forall (A : Type) (eqA : forall (_ : A) (_ : A), Prop) (l : list A), SetoidList.inclA eqA nil l

TX: reduction_t A X

UExp: forall (X Y : Type) (_ : function X Y) (_ : relation2 X Y) (_ : nat), relation2 X Y

UExp_inc: forall (X Y : Type) (F : function X Y) (_ : increasing F) (n : nat) (R S : relation2 X Y) (_ : incl R S), incl (UExp F R n) (UExp F S n)

UExp_incl: forall (X Y : Type) (F : function X Y) (R : relation2 X Y) (n : nat), incl (UExp F R n) (UExp F R (S n))

UExp_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : monotonic TX TY F) (n : nat), monotonic TX TY (fun R : relation2 X Y => UExp F R n)

UIter: forall (X Y : Type) (_ : function X Y) (_ : relation2 X Y), relation2 X Y

UIter_incl: forall (X Y : Type) (F : function X Y) (R : relation2 X Y), incl R (UIter F R)

UIter_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : monotonic TX TY F), monotonic TX TY (UIter F)

Union2_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F G : function X Y) (_ : monotonic TX TY F) (_ : monotonic TX TY G), monotonic TX TY (Union2 F G)

Union_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (I : Type) (H : forall _ : I, function X Y) (_ : forall i : I, monotonic TX TY (H i)), monotonic TX TY (Union H)

Weak: forall (A X : Type) (_ : reduction_t A X), reduction_t A X

ZMicromega.Vars.Facts.MP.double_inclusion: forall s1 s2 : FSetPositive.PositiveSet.t, iff (FSetPositive.PositiveSet.Equal s1 s2) (and (FSetPositive.PositiveSet.Subset s1 s2) (FSetPositive.PositiveSet.Subset s2 s1))

chaining_l_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (E : relation2 X X) (_ : expansion1 TX TX E), monotonic TX TY (chaining_l E)

chaining_r_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (T : relation2 Y Y) (_ : simulation TY TY T), monotonic TX TY (chaining_r T)

comp: forall (X Y Z : Type) (_ : relation2 X Y) (_ : relation2 Y Z), relation2 X Z

comp_incl: forall (X Y Z : Type) (R' : relation2 X Y) (S' : relation2 Y Z) (R : relation2 X Y) (S : relation2 Y Z) (_ : incl R R') (_ : incl S S'), incl (comp R S) (comp R' S')

constant_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (R : relation2 X Y) (_ : simulation TX TY R), monotonic TX TY (constant R)

eta2: forall (X Y : Type) (_ : relation2 X Y), relation2 X Y

evolve_incl: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (l : Lbl A) (S R S' : relation2 X Y) (_ : incl S S') (_ : evolve_1 TX TY l R S), evolve_1 TX TY l R S'

evolve_t: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : relation2 X Y) (_ : relation2 X Y), Prop

expand_wexpand: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y), incl (expand TX TY) (wexpand TX TY)

identity_mon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y), monotonic TX TY (identity (Y:=Y))

incl_r: forall A X : Type, relation (reduction A X)

mkmon: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : increasing F) (_ : forall (R S : relation2 X Y) (_ : evolve_t TX TY R S) (_ : incl R S), evolve_t TX TY (F R) (F S)) (_ : forall (R S : relation2 X Y) (_ : evolve TX TY R S) (_ : incl R S), evolve_a TX TY (F R) (F S)), monotonic TX TY F

mon_t: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : monotonic TX TY F) (R S : relation2 X Y) (_ : evolve_t TX TY R S) (_ : incl R S), evolve_t TX TY (F R) (F S)

monotonic: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : function X Y), Prop

monotonic_correct: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : monotonic TX TY F) (R : relation2 X Y) (_ : evolve TX TY R (F R)), simulation TX TY (UIter F R)

monotonic_wmonotonic: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : monotonic TX TY F), wmonotonic TX TY F

plus_incl: forall (X : Type) (T' T : relation X) (_ : incl T T'), incl (plus T) (plus T')

reduction_t: forall (_ : Type) (_ : Type), Type

relation2: forall (_ : Type) (_ : Type), Type

star_incl: forall (X : Type) (T' T : relation X) (_ : incl T T'), incl (star T) (star T')

trans: forall (X Y : Type) (_ : relation2 X Y), relation2 Y X

trans_incl: forall (X Y : Type) (R' R : relation2 X Y) (_ : incl R R'), incl (trans R) (trans R')

union2: forall (X Y : Type) (_ : relation2 X Y) (_ : relation2 X Y), relation2 X Y

union2_incl: forall (X Y : Type) (R' R1' R R1 : relation2 X Y) (_ : incl R R') (_ : incl R1 R1'), incl (union2 R R1) (union2 R' R1')

union: forall (X Y I : Type) (_ : forall (_ : I) (_ : X) (_ : Y), Prop), relation2 X Y

wexpand_bisim: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y), incl (wexpand TX TY) (bisim TX TY)

wmonotonic: forall (A X Y : Type) (_ : reduction_t A X) (_ : reduction_t A Y) (_ : function X Y), Prop

wmonotonic_correct: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (R : relation2 X Y) (_ : simulation_t TX TY R) (_ : evolve_a TX TY R (F R)), simulation TX TY (UIter F R)

wmonotonic_correct_t: forall (A X Y : Type) (TX : reduction_t A X) (TY : reduction_t A Y) (F : function X Y) (_ : wmonotonic TX TY F) (R : relation2 X Y) (_ : simulation_t TX TY R), simulation_t TX TY (UIter F R)"""
    )
]

In [ ]:
SPLIT_PREMISES = [
    premises.split("\n\n") for _,_,premises in DATASET
]

SPLIT_PREMISES[0][:5]

In [ ]:
for i, (proposition, proof_state, premises) in enumerate(DATASET):
    pprint(proof_state)

## ranking

In [ ]:
RANKING_SYSTEM_PROMPT = """You are an expert at proving theorems with the Coq theorem prover.
You will be shown a proposition, a proof state, and a list of potentially useful lemmas.
Your task is to rank these lemmas in order of how useful they are for making *immediate* progress on the proof.
Lemmas should be ranked higher if they are needed right now, and lower if they are not needed right now, regardless of whether they are likely to be useful later.
Make sure you include all lemmas from the list in your ranking, and that you do not include any lemmas that are not in the list."""

RANKING_TOOL = OpenaiTool(
    name="rank_lemmas",
    description="Rank lemmas in order of how useful they are for making immediate progress on the proof.",
    parameters={
        "type": "object",
        "properties": {
            "lemma_identifiers": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "The identifiers of the lemmas to rank, ordered from most useful to least useful."
                }
            }
        }
    }
)

In [ ]:
TEST_PROP, TEST_PROOF_STATE, TEST_PREMISES = DATASET[0]

TEST_USER_MESSAGE = f"""[PROPOSITION]
{TEST_PROP}

[PROOF STATE]
{TEST_PROOF_STATE}

[PREMISES]
{TEST_PREMISES}"""

pprint(TEST_USER_MESSAGE)

In [ ]:
if os.path.exists("./TEST_RESULT.pkl"):
    with open("./TEST_RESULT.pkl", "rb") as f:
        TEST_RESULT = pickle.load(f)
        print("Loaded previous test result")
else:
    TEST_RESULT = chat_with_tools(
        [
            SystemMessage(content=RANKING_SYSTEM_PROMPT),
            UserMessage(content=TEST_USER_MESSAGE),
        ],
        OpenaiChatPromptConfig(n=5),
        OpenaiToolConfig(
            tools=[RANKING_TOOL],
            tool_choice=RANKING_TOOL
        )
    )

pprint(TEST_RESULT)

In [ ]:
# pickle the test result
with open("TEST_RESULT.pkl", "wb") as f:
    pickle.dump(TEST_RESULT, f)

In [ ]:
pprint([call[0].arguments['lemma_identifiers'] for call in TEST_RESULT[0]])

In [ ]:
pprint([len(call[0].arguments['lemma_identifiers']) for call in TEST_RESULT[0]])
pprint(len(SPLIT_PREMISES[0]))

In [ ]:
def get_rankings(proposition: str, proof_state: str, premises: t.List[str], n: int = 5) -> t.List[t.List[str]]:
    premise_shufflings = []
    for _ in range(n):
        shuffled_premises = premises.copy()
        random.shuffle(shuffled_premises)
        premise_shufflings.append('\n\n'.join(shuffled_premises))
    user_messages = [
        f"""[PROPOSITION]
{proposition}

[PROOF STATE]
{proof_state}

[PREMISES]
{premises}"""
        for premises in premise_shufflings
    ]

    ans: t.List[t.List[str]] = []
    for user_message in user_messages:
        result = chat_with_tools(
            [
                SystemMessage(content=RANKING_SYSTEM_PROMPT),
                UserMessage(content=user_message)
            ],
            OpenaiChatPromptConfig(),
            OpenaiToolConfig(
                tools=[RANKING_TOOL],
                tool_choice=RANKING_TOOL
            )
        )
        ans.append([call[0].arguments['lemma_identifiers'] for call in result[0]])
    return ans

In [ ]:
RANKINGS = []

# if ./RANKINGS.pkl exists, load it into RANKINGS
if os.path.exists('./RANKINGS.pkl'):
    with open('./RANKINGS.pkl', 'rb') as f:
        RANKINGS = pickle.load(f)
        print(f"Loaded RANKINGS from ./RANKINGS.pkl")
else:
    for prop, premises in zip(DATASET, SPLIT_PREMISES):
        print(f"Ranking lemmas for proposition {prop}...")
        rankings = get_rankings(prop[0], prop[1], premises)
        RANKINGS.append(rankings)
    

In [ ]:
# pickle rankings to ./RANKINGS.pkl
with open('RANKINGS.pkl', 'wb') as f:
    pickle.dump(RANKINGS, f)

In [ ]:
pprint(RANKINGS[0])

In [ ]:
# the schulze method is an O(n^3) algorithm to combine differnt rankings into a single universal ranking
# it's generally used in ranked choice voting systems, but we can use it here to combine the rankings from different gpt calls
from src.one_off_experments.schulze import compute_ranks

def list_to_ranking(lst: t.List[str]) -> t.Tuple[t.List[t.List[str]], int]:
    return [[item] for item in lst], 1

FLAT_RANKINGS = []

for rankingses in RANKINGS:
    FLAT_RANKINGS.append([list_to_ranking(ranking) 
    for rankings in rankingses for ranking in rankings])

FLAT_RANKINGS[0]

In [ ]:
COMBINED_RANKINGS = []
for rankings in FLAT_RANKINGS:
    candidate_names = list(set([item[0] for ranking in rankings for item in ranking[0]]))
    combined_ranking = compute_ranks(candidate_names, rankings)
    COMBINED_RANKINGS.append(combined_ranking)

pprint(COMBINED_RANKINGS[0])

In [ ]:
def ranking_to_list(ranking: t.List[t.List[str]]) -> t.List[str]:
    return [item for items in ranking for item in items]

FLAT_COMBINED_RANKINGS = [ranking_to_list(ranking) for ranking in COMBINED_RANKINGS]

pprint(FLAT_COMBINED_RANKINGS)

## Pairwise ranking
- for each premise, sample k other premises and put them head to head.
- rank by number of wins
- reasoning/no reasoning
- gpt-3.5 vs. gpt-4

In [ ]:
PAIRWISE_SYSTEM_PROMPT = """You are an expert at proving theorems with the Coq theorem prover.
You will be shown a proposition, a proof state, and two potentially useful lemmas (A and B).
Your task is to decide whether A or B is more useful for making *immediate* progress on the proof.
A lemma should be considered more useful if it is needed right now, and less useful if it is not needed right now, regardless of whether it is likely to be useful later.
So for example, if one lemma is necessary to make progress right now, and the other might be useful later, you should choose the lemma that is necessary right now."""

PAIRWISE_TOOL = OpenaiTool(
    name="rank_lemmas",
    description="Decide whether lemma A or B is more useful for making immediate progress on the proof.",
    parameters={
        "type": "object",
        "properties": {
            "decision": {
                "type": "string",
                "enum": ["A", "B"],
                "description": "The lemma that is more useful for making immediate progress on the proof."
            }
        }
    }
)

In [ ]:
TEST_PROP, TEST_PROOF_STATE, TEST_PREMISES = DATASET[0]
TEST_PREMISE_A = random.choice(SPLIT_PREMISES[0])
TEST_PREMISE_B = SPLIT_PREMISES[0][3]

print("test prop:", TEST_PROP)
print("test proof state:", TEST_PROOF_STATE)
print("test premise A:", TEST_PREMISE_A)
print("test premise B:", TEST_PREMISE_B)

In [ ]:
TEST_USER_MESSAGE = f"""[PROPOSITION]
{TEST_PROP}

[PROOF STATE]
{TEST_PROOF_STATE}

[PREMISE A]
{TEST_PREMISE_A}

[PREMISE B]
{TEST_PREMISE_B}"""

pprint(TEST_USER_MESSAGE)

In [ ]:
if os.path.exists("./PAIRWISE_TEST_RESULT.pkl"):
    with open("./PAIRWISE_TEST_RESULT.pkl", "rb") as f:
        PAIRWISE_TEST_RESULT = pickle.load(f)
        print("Loaded previous test result")
else:
    PAIRWISE_TEST_RESULT = chat_with_tools(
        [
            SystemMessage(content=PAIRWISE_SYSTEM_PROMPT),
            UserMessage(content=TEST_USER_MESSAGE),
        ],
        OpenaiChatPromptConfig(n=5),
        OpenaiToolConfig(
            tools=[PAIRWISE_TOOL],
            tool_choice=PAIRWISE_TOOL
        )
    )

pprint(PAIRWISE_TEST_RESULT)

with open("PAIRWISE_TEST_RESULT.pkl", "wb") as f:
    pickle.dump(PAIRWISE_TEST_RESULT, f)

In [ ]:
pprint([call[0].arguments['decision'] for call in PAIRWISE_TEST_RESULT[0]])

In [ ]:
def pairwise_rank(proposition: str, proof_state: str, premise_a: str, premise_b: str, k: int = 5) -> t.Tuple[float, float]:
    user_message = f"""[PROPOSITION]
{proposition}

[PROOF STATE]
{proof_state}

[PREMISE A]
{premise_a}

[PREMISE B]
{premise_b}"""

    user_message_reversed = f"""[PROPOSITION]
{proposition}

[PROOF STATE]
{proof_state}

[PREMISE A]
{premise_b}

[PREMISE B]
{premise_a}"""

    result = chat_with_tools(
        [
            SystemMessage(content=PAIRWISE_SYSTEM_PROMPT),
            UserMessage(content=user_message),
        ],
        OpenaiChatPromptConfig(n = k // 2),
        OpenaiToolConfig(
            tools=[PAIRWISE_TOOL],
            tool_choice=PAIRWISE_TOOL
        )
    )

    result_reversed = chat_with_tools(
        [
            SystemMessage(content=PAIRWISE_SYSTEM_PROMPT),
            UserMessage(content=user_message_reversed),
        ],
        OpenaiChatPromptConfig(n = k - k // 2),
        OpenaiToolConfig(
            tools=[PAIRWISE_TOOL],
            tool_choice=PAIRWISE_TOOL
        )
    )

    a_wins = sum([call[0].arguments['decision'] == "A" for call in result[0]]) + sum([call[0].arguments['decision'] == "B" for call in result_reversed[0]])
    b_wins = sum([call[0].arguments['decision'] == "B" for call in result[0]]) + sum([call[0].arguments['decision'] == "A" for call in result_reversed[0]])

    return a_wins / k, b_wins / k

In [ ]:
pairwise_rank(TEST_PROP, TEST_PROOF_STATE, TEST_PREMISE_A, TEST_PREMISE_B)

In [ ]:
def pairwise_rank_all_premises(proposition: str, proof_state: str, premises: t.List[str], k: int = 5, num_pairs_per_item = 5) -> t.List[t.Tuple[float, float]]:
    scores: t.Dict[int, float] = {}
    for i in range(len(premises)):
        print(f"Ranking premise {i} of {len(premises)}...")
        for _ in range(num_pairs_per_item):
            j = random.choice([x for x in range(len(premises)) if x != i])
            a, b = pairwise_rank(proposition, proof_state, premises[i], premises[j], k)
            scores[i] = scores.get(i, 0) + a
            scores[j] = scores.get(j, 0) + b
    total_scores = sum(scores.values())
    normalized_scores = {k: v / total_scores for k, v in scores.items()}

    # return premises ordered by their normalized scores
    return sorted(normalized_scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
PAIRWISE_RANKINGS = []

if os.path.exists("./PAIRWISE_RANKINGS.pkl"):
    with open("./PAIRWISE_RANKINGS.pkl", "rb") as f:
        PAIRWISE_RANKINGS = pickle.load(f)
        print("Loaded previous pairwise rankings")
else:
    for i, ((proposition, proof_state, _), premises) in enumerate(zip(DATASET, SPLIT_PREMISES)):
        print(f"Ranking premises for proposition {proposition}...")
        pairwise_rankings = pairwise_rank_all_premises(proposition, proof_state, premises)
        PAIRWISE_RANKINGS.append(pairwise_rankings)

    with open("PAIRWISE_RANKINGS.pkl", "wb") as f:
        pickle.dump(PAIRWISE_RANKINGS, f)

pprint(PAIRWISE_RANKINGS[0])

In [ ]:
RANKED_PREMISES = []

for rankings, premises in zip(PAIRWISE_RANKINGS, SPLIT_PREMISES):
    ranked_premises = [premises[i] for i, _ in rankings]
    RANKED_PREMISES.append(ranked_premises)

pprint(RANKED_PREMISES)